# Modèle de langage et génération de texte

Dans ces travaux pratiques, nous allons apprendre à un réseau récurrent à écrire « à la manière de Voltaire », caractère par caractère : à partir des 30 caractères précédents, le modèle prédit la probabilité de chaque caractère suivant. C'est un modèle de langage, ici à l'échelle des caractères.

1. Un modèle de langage par caractères.
2. Générer du texte, avec une température.
3. L'entraînement, en regardant ce que le modèle écrit.
4. L'effet de la température.
5. LSTM, GRU ou RNN simple ?
6. Deux couches de LSTM.

Ce TP a besoin d'une carte graphique : vérifiez dans `Exécution > Modifier le type d'exécution` que l'accélérateur matériel est un GPU. Sur un processeur, ses entraînements prendraient plusieurs heures.

## Récupération des données

Les textes de Voltaire du corpus viennent de [gutenberg.org](https://www.gutenberg.org), sans leurs en-têtes, licences et préfaces : notre modèle de langage ne doit pas apprendre à écrire les mentions légales de Gutenberg ou les préfaces des éditeurs. Nous n'utilisons que `voltaire/voltaire_clean.txt`, qui les réunit. La cellule en affiche les 1 000 premiers caractères.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh voltaire
!ls -l voltaire/
!head -c 1000 voltaire/voltaire_clean.txt

## Installation et import des bibliothèques

In [ ]:
!pip install -q lightning torchmetrics

In [ ]:
import pathlib
from collections.abc import Sequence

import lightning
import matplotlib.pyplot as plt
import numpy as np
import pandas
import torch
import torchmetrics
from lightning.pytorch.callbacks import Callback, TQDMProgressBar
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.utilities.model_summary import ModelSummary
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# Mêmes tirages aléatoires à chaque exécution : des résultats reproductibles
lightning.seed_everything(0)

## Préparation des données

La cellule suivante prépare les exemples :

- le texte, en minuscules, dans `text` ;
- le vocabulaire : les caractères du texte, triés, dans `index_to_char` (indice → caractère), et `char_to_index` (caractère → indice) ; `encode` transforme une chaîne en tenseur d'indices ;
- les exemples : des séquences de 30 caractères (`maxlen`), prises tous les 3 caractères ; la cible de chacune est le caractère qui la suit ;
- `X`, les séquences encodées, de forme `(nombre de séquences, 30)`, et `y`, les indices des caractères suivants.

In [ ]:
text = pathlib.Path("voltaire/voltaire_clean.txt").read_text(encoding="utf8").lower()
index_to_char = sorted(set(text))
char_to_index = {char: index for index, char in enumerate(index_to_char)}
print(f"{len(text)} caractères : {text[:80]}…")
print(f"{len(index_to_char)} caractères différents : {''.join(index_to_char)}")


def encode(characters: str) -> torch.Tensor:
  return torch.tensor([char_to_index[char] for char in characters])


maxlen = 30
step = 3
sentences = []
next_chars = []
for i in range(0, len(text) - maxlen, step):
  sentences.append(text[i : i + maxlen])
  next_chars.append(text[i + maxlen])

X = torch.stack([encode(sentence) for sentence in sentences])
y = encode(next_chars)
print(f"Formes de X et y : {tuple(X.shape)}, {tuple(y.shape)}")
print(f"Premier exemple : « {sentences[0]} » → « {next_chars[0]} »")

## 1. Un modèle de langage par caractères

Écrivez la classe `CharRNN`, un `nn.Module` :

- `__init__(cell=nn.LSTM, hidden_dim=128, num_layers=1)` crée ses couches :
  - une couche `nn.Embedding` : un vecteur de 8 nombres pour chaque caractère du vocabulaire ;
  - la couche récurrente `cell`, une classe parmi `nn.LSTM`, `nn.GRU` et `nn.RNN` : `num_layers` couches empilées de `hidden_dim` unités, qui lisent les vecteurs des caractères ;
  - une couche `nn.Linear`, qui donne un score (un logit) à chaque caractère du vocabulaire ;
- `forward(sentences)` reçoit des séquences d'indices, de forme `(batch, longueur)`, et renvoie les logits du caractère qui suit chacune, de forme `(batch, len(index_to_char))`, calculés à partir de la sortie de la couche récurrente au dernier pas de temps.

Pas de softmax à la fin : la perte `cross_entropy` du `LanguageModel` (fourni après la solution) part des logits.

Les parties suivantes l'appellent avec la cellule à comparer, d'abord `CharRNN()`, un LSTM de 128 unités.

**Documentation utile**

- [`torch.nn.LSTM`](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html)

In [ ]:
class CharRNN(nn.Module):
  def __init__(
    self, cell: type[nn.RNNBase] = nn.LSTM, hidden_dim: int = 128, num_layers: int = 1
  ) -> None:
    super().__init__()
    # Votre code ici

  def forward(self, sentences: torch.Tensor) -> torch.Tensor:
    # Votre code ici
    ...

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `nn.Embedding(len(index_to_char), 8)`
- `cell(8, hidden_dim, num_layers=num_layers, batch_first=True)` : `nn.LSTM`, `nn.GRU` et `nn.RNN` prennent les mêmes paramètres ; la taille des vecteurs d'entrée, 8, d'abord ; `batch_first=True` pour des tenseurs de forme `(batch, longueur, 8)`
- la couche récurrente renvoie deux valeurs : sa sortie à chaque pas de temps, de forme `(batch, longueur, hidden_dim)`, puis ses derniers états (un tuple `(h_n, c_n)` pour un LSTM) ; `outputs[:, -1]` est la sortie du dernier pas, de forme `(batch, hidden_dim)`
- `nn.Linear(hidden_dim, len(index_to_char))`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
class CharRNN(nn.Module):
  def __init__(
    self, cell: type[nn.RNNBase] = nn.LSTM, hidden_dim: int = 128, num_layers: int = 1
  ) -> None:
    super().__init__()
    self.embedding = nn.Embedding(..., ...)
    self.rnn = cell(..., hidden_dim, num_layers=num_layers, batch_first=True)
    self.head = nn.Linear(..., ...)

  def forward(self, sentences: torch.Tensor) -> torch.Tensor:
    outputs, _ = self.rnn(self.embedding(sentences))
    return self.head(outputs[...])
```

### Solution

In [ ]:
class CharRNN(nn.Module):
  """Embed the characters, run a recurrent network, score the next one."""

  def __init__(
    self, cell: type[nn.RNNBase] = nn.LSTM, hidden_dim: int = 128, num_layers: int = 1
  ) -> None:
    super().__init__()
    self.embedding = nn.Embedding(len(index_to_char), 8)
    self.rnn = cell(8, hidden_dim, num_layers=num_layers, batch_first=True)
    self.head = nn.Linear(hidden_dim, len(index_to_char))

  def forward(self, sentences: torch.Tensor) -> torch.Tensor:
    # Sortie de la dernière couche à chaque pas : (batch, longueur, hidden_dim)
    outputs, _ = self.rnn(self.embedding(sentences))
    # Le dernier pas a lu toute la séquence : il prédit le caractère suivant
    return self.head(outputs[:, -1])

La cellule suivante enveloppe un `CharRNN` dans un `LanguageModel`, un `LightningModule` : la perte d'entropie croisée, la justesse (la proportion de caractères suivants bien prédits), l'optimiseur Adam. Elle construit ensuite le modèle du TP, un LSTM de 128 unités, et en affiche le résumé.

In [ ]:
class LanguageModel(lightning.LightningModule):
  """Lightning wrapper: predict the character following a sequence."""

  def __init__(self, model: nn.Module, learning_rate: float = 1e-3) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["model"])
    self.model = model
    # Une entrée d'exemple : le résumé du modèle affiche les formes des sorties
    self.example_input_array = torch.zeros(1, maxlen, dtype=torch.long)
    # torchmetrics demande une instance de métrique par étape
    self.accuracies = nn.ModuleDict(
      {
        f"{stage}_accuracy": torchmetrics.Accuracy(
          task="multiclass", num_classes=len(index_to_char)
        )
        for stage in ("train", "val")
      }
    )

  def forward(self, sentences: torch.Tensor) -> torch.Tensor:
    return self.model(sentences)

  def _step(self, batch: tuple[torch.Tensor, torch.Tensor], stage: str) -> torch.Tensor:
    sentences, next_chars = batch
    logits = self(sentences)
    loss = nn.functional.cross_entropy(logits, next_chars)
    accuracy = self.accuracies[f"{stage}_accuracy"]
    accuracy(logits, next_chars)
    self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
    self.log(f"{stage}_accuracy", accuracy, on_step=False, on_epoch=True, prog_bar=True)
    return loss

  def training_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self._step(batch, "train")

  def validation_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self._step(batch, "val")

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)


model = LanguageModel(CharRNN())
print(ModelSummary(model, max_depth=2))

## 2. Générer du texte, avec une température

Écrivez la fonction `generate(model, seed, n=200, temperature=1.0)`, qui renvoie le texte `seed` suivi de `n` caractères générés par `model`, un à un :

1. encodez les `maxlen` derniers caractères du texte, et donnez-les au modèle sous la forme d'un batch d'une seule séquence ;
2. le modèle renvoie les logits $z_i$ du caractère suivant ; divisez-les par la température $T$ avant le softmax : $p_i \propto \exp(z_i / T)$ ;
3. tirez le caractère suivant selon ces probabilités, et ajoutez-le au texte.

Avec $T = 1$, on tire selon les probabilités du modèle ; une température plus basse favorise les caractères les plus probables, une plus haute aplatit la distribution.

La cellule qui suit la solution l'essaie sur le modèle pas encore entraîné.

**Documentation utile**

- [`torch.multinomial`](https://docs.pytorch.org/docs/stable/generated/torch.multinomial.html)
- [`torch.no_grad`](https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html)

In [ ]:
@torch.no_grad()
def generate(
  model: lightning.LightningModule, seed: str, n: int = 200, temperature: float = 1.0
) -> str:
  # Votre code ici
  ...

### Conseils supplémentaires

Les appels à utiliser :

- `@torch.no_grad()`, déjà sur la fonction : pas de calcul de gradient pendant la génération ; `model.eval()` met le modèle en mode évaluation, et `model.train(training)` lui rend à la fin le mode qu'il avait (`training = model.training`) : Lightning n'y remet pas un modèle passé en évaluation avant `fit`
- `generated[-maxlen:]` : les `maxlen` derniers caractères ; `encode(...)[None, :]` ajoute la dimension du batch, de taille 1 ; `.to(model.device)` place le tenseur sur le processeur ou la carte graphique, là où est le modèle
- `model(x)` renvoie les logits, de forme `(1, len(index_to_char))` : `[0]` garde ceux de l'unique séquence
- `(logits / temperature).softmax(dim=-1)` : les probabilités, à la température `temperature`
- `torch.multinomial(probas, 1)` tire un indice selon les probabilités `probas`, et `int(...)` en fait un entier Python

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
@torch.no_grad()
def generate(
  model: lightning.LightningModule, seed: str, n: int = 200, temperature: float = 1.0
) -> str:
  training = model.training
  model.eval()
  generated = seed
  for _ in range(n):
    x = encode(generated[...])[None, :].to(model.device)
    logits = model(x)[0]
    probas = (logits / ...).softmax(dim=-1)
    generated += index_to_char[int(torch.multinomial(..., 1))]
  model.train(training)
  return generated
```

### Solution

In [ ]:
@torch.no_grad()
def generate(
  model: lightning.LightningModule, seed: str, n: int = 200, temperature: float = 1.0
) -> str:
  # Le mode du modèle, rendu à la fin : Lightning ne remet pas en mode
  # entraînement un modèle laissé en évaluation avant fit
  training = model.training
  model.eval()
  generated = seed
  for _ in range(n):
    # Les maxlen derniers caractères, en un batch d'une séquence
    x = encode(generated[-maxlen:])[None, :].to(model.device)
    logits = model(x)[0]
    # p_i ∝ exp(z_i / T)
    probas = (logits / temperature).softmax(dim=-1)
    generated += index_to_char[int(torch.multinomial(probas, 1))]
  model.train(training)
  return generated

Avant l'entraînement, le modèle donne à peu près la même probabilité à tous les caractères : il écrit n'importe quoi.

In [ ]:
print(generate(model, text[:maxlen], n=100))

## 3. Entraîner le modèle

La fonction `train` entraîne un modèle avec Lightning, et renvoie ses métriques de chaque epoch, lues dans le journal du `CSVLogger` :

- des batchs de 512 séquences ; la norme du gradient est écrêtée à 1 (`gradient_clip_val`), contre l'explosion du gradient, fréquente dans les RNN ;
- la fin du corpus, 10 % des séquences, sert à la validation. Un découpage aléatoire mettrait presque les mêmes séquences des deux côtés, puisque deux séquences voisines ont 27 caractères en commun.

Le callback `SamplingCallback` fait écrire 200 caractères au modèle toutes les `every` epochs, à partir d'un extrait du texte tiré au hasard : on voit ainsi le modèle progresser.

L'entraînement du modèle dure 20 epochs.

In [ ]:
n_train = int(len(X) * 0.9)
train_loader = DataLoader(
  TensorDataset(X[:n_train], y[:n_train]), batch_size=512, shuffle=True
)
val_loader = DataLoader(TensorDataset(X[n_train:], y[n_train:]), batch_size=2048)


def train(
  model: LanguageModel,
  name: str,
  max_epochs: int = 20,
  callbacks: Sequence[Callback] = (),
) -> pandas.DataFrame:
  trainer = lightning.Trainer(
    max_epochs=max_epochs,
    accelerator="auto",
    devices=1,
    gradient_clip_val=1.0,  # écrêtage de la norme du gradient
    logger=CSVLogger("logs", name=name),
    # La barre de progression de tqdm : celle de rich, que Lightning choisit
    # quand rich est installé (Colab), ne laisse pas un callback écrire avec print
    callbacks=[TQDMProgressBar(), *callbacks],
    enable_checkpointing=False,
    enable_model_summary=False,
  )
  trainer.fit(model, train_loader, val_loader)
  # Métriques de chaque epoch, écrites par le CSVLogger sur des lignes
  # différentes pour l'entraînement et la validation : on les regroupe
  return (
    pandas.read_csv(pathlib.Path(trainer.logger.log_dir) / "metrics.csv")
    .groupby("epoch")
    .mean(numeric_only=True)
  )


class SamplingCallback(Callback):
  def __init__(self, every: int = 5, n: int = 200) -> None:
    super().__init__()
    self.every = every
    self.n = n

  def on_train_epoch_end(
    self, trainer: lightning.Trainer, pl_module: lightning.LightningModule
  ) -> None:
    epoch = trainer.current_epoch + 1
    if epoch % self.every == 0:
      start = np.random.randint(len(text) - maxlen)
      seed = text[start : start + maxlen]
      print(f"\nAprès {epoch} epochs, à partir de « {seed} » :")
      # pl_module : le modèle que le Trainer entraîne
      print(generate(pl_module, seed, n=self.n))


history = train(model, "lstm", callbacks=[SamplingCallback(every=5)])
print(
  f"Après 20 epochs : perte de validation {history['val_loss'].iloc[-1]:.3f}, "
  f"justesse de validation {history['val_accuracy'].iloc[-1]:.3f}"
)

## 4. L'effet de la température

Avec le modèle entraîné, générez 300 caractères à partir du même début de phrase, par exemple `"candide était "`, avec les températures 0,2, 0,5, 1 et 1,5. Comment le texte change-t-il quand la température baisse ? quand elle monte ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
for temperature in [0.2, 0.5, 1.0, 1.5]:
  print(f"Température {temperature} :")
  print(generate(model, "candide était ", n=..., temperature=...))
  print()
```

### Solution

In [ ]:
for temperature in [0.2, 0.5, 1.0, 1.5]:
  print(f"Température {temperature} :")
  print(generate(model, "candide était ", n=300, temperature=temperature))
  print()

À basse température (0,2), le modèle choisit presque toujours le caractère le plus probable : il écrit des mots courts et fréquents, presque tous réels (« de la », « les personnes », « la terre »), et tourne vite en boucle sur les mêmes groupes de mots. À haute température (1,5), la distribution s'aplatit : les caractères peu probables sortent plus souvent, et le texte n'est plus qu'une suite de syllabes, de mots inventés et de signes (« _ », « ; », « ! », des chiffres). Entre les deux, vers 0,5, le texte est le plus vraisemblable : des mots plus variés, presque tous français, et moins de répétitions ; à 1, les mots inventés sont déjà nombreux.

Aucun de ces textes ne veut rien dire : après 20 epochs, le modèle ne prédit juste le caractère suivant qu'une fois sur deux environ (52 % de justesse de validation). Il a appris les mots courts, des terminaisons françaises (« -ait », « -ent ») et la ponctuation, pas encore les mots longs ni le sens. Les textes changent d'une exécution à l'autre, puisque les caractères sont tirés au hasard, mais ces différences entre températures restent.

## 5. LSTM, GRU ou RNN simple ?

Entraînez trois modèles de 128 unités, un LSTM, un GRU et un RNN simple (`nn.RNN`), pendant 10 epochs chacun, et comparez leurs pertes de validation (`val_loss`), par exemple sur un même graphique. Quelle cellule fait le mieux ? Le RNN simple est-il en retard, comme la disparition du gradient le laisserait attendre ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `LanguageModel(CharRNN(cell=nn.GRU))` : le même modèle, avec un GRU
- `train(model, name, max_epochs=10)` renvoie un tableau `pandas.DataFrame`, une ligne par epoch (son index, à partir de 0), dont la colonne `"val_loss"` est la perte de validation
- n'appelez pas `history` la variable de la boucle : le `history` de la partie 3 sert encore dans la partie 6

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
histories = {}
for name, cell in [("LSTM", nn.LSTM), ("GRU", ...), ("RNN", ...)]:
  histories[name] = train(LanguageModel(CharRNN(cell=...)), name, max_epochs=...)

for name, cell_history in histories.items():
  plt.plot(cell_history.index + 1, cell_history[...], label=name)
plt.xlabel("Epoch")
plt.ylabel(...)
plt.legend()
plt.show()
```

### Solution

In [ ]:
histories = {}
for name, cell in [("LSTM", nn.LSTM), ("GRU", nn.GRU), ("RNN", nn.RNN)]:
  histories[name] = train(LanguageModel(CharRNN(cell=cell)), name, max_epochs=10)

# cell_history et non history : celui de la partie 3 sert encore
for name, cell_history in histories.items():
  plt.plot(cell_history.index + 1, cell_history["val_loss"], label=name)
  print(f"{name} : perte de validation minimale {cell_history['val_loss'].min():.3f}")
plt.xlabel("Epoch")
plt.ylabel("Perte de validation")
plt.legend()
plt.show()

Après 10 epochs, le GRU est nettement devant : une perte de validation minimale de 1,58, mieux que le LSTM de la partie 3 après 20 epochs (1,60). Le RNN simple (1,70) devance même le LSTM (1,77). Aucune des trois courbes n'a fini de descendre : ce classement dit surtout quelle cellule apprend le plus vite. Le LSTM a le plus de poids à régler (70 656 dans sa couche récurrente, contre 52 992 pour le GRU et 17 664 pour le RNN simple), et il démarre plus lentement ; il rattrape une partie de son retard avec plus d'epochs.

Le RNN simple n'est pas en retard ici : 30 caractères forment une séquence courte, et la disparition du gradient (il s'atténue d'un pas de temps à l'autre) ne le gêne pas encore beaucoup. Elle le pénalise sur des séquences plus longues, où les portes du LSTM et du GRU laissent passer l'information, et le gradient, sur de plus longues distances. L'écrêtage du gradient (`gradient_clip_val`) le protège de l'autre risque, l'explosion du gradient.

Ces modèles n'ont pas encore surappris : leur perte de validation baisse toujours, et reste proche de celle d'entraînement. Avec beaucoup plus d'epochs, ou un modèle plus gros, elle finirait par remonter alors que celle d'entraînement baisse encore : ce serait le moment d'arrêter l'entraînement (callback [`EarlyStopping`](https://lightning.ai/docs/pytorch/stable/common/early_stopping.html) de Lightning).

## 6. Deux couches de LSTM

Un dernier modèle, donné : deux LSTM empilés (`num_layers=2`). La première couche transmet sa sortie à chaque pas de temps, la séquence que lit la seconde. Comparez sa perte de validation à celle du modèle de la partie 3, et ce qu'il écrit.

In [ ]:
deep_model = LanguageModel(CharRNN(num_layers=2))
print(ModelSummary(deep_model, max_depth=2))
deep_history = train(deep_model, "deep_lstm", callbacks=[SamplingCallback(every=10)])
print(
  f"Perte de validation minimale : un LSTM {history['val_loss'].min():.3f}, "
  f"deux {deep_history['val_loss'].min():.3f}"
)

Les deux couches font mieux que le LSTM seul de la partie 3 : après 20 epochs, une perte de validation minimale de 1,50 contre 1,60, et mieux que le GRU de la partie 5 en 10 epochs (1,58). Elles coûtent 2,6 fois plus de paramètres (environ 212 000 contre 80 400) et des epochs plus longues. Le texte, lui, reste fait de mots courts justes et de mots longs inventés : pour écrire des phrases, un modèle de langage par caractères a besoin de bien plus d'entraînement.